In [58]:
import warnings

# Try importing UnitsWarning from the new location
try:
    from astropy.units import UnitsWarning
except ImportError:
    # fallback for older astropy versions
    from astropy.utils.exceptions import UnitsWarning

warnings.filterwarnings("ignore", category=UnitsWarning)
import pandas as pd
import numpy as np
import requests
import io
from alerce.core import Alerce

alerce = Alerce()

# -----------------------------------------------------------
# 1) Load the table from the paper
# -----------------------------------------------------------
def load_table(url):
    txt = requests.get(url).text
    colspecs = [(0,3), (4,21), (22,34), (35,46), (47,52), (53,91), (92,101)]
    names = ['Rank','ID','RAdeg','DEdeg','-logp','Label','Ref']
    df = pd.read_fwf(io.StringIO(txt), colspecs=colspecs, names=names, skiprows=3)
    df['RAdeg'] = pd.to_numeric(df['RAdeg'], errors='coerce')
    df['DEdeg'] = pd.to_numeric(df['DEdeg'], errors='coerce')
    df['Label'] = df['Label'].fillna("").astype(str)
    return df

# -----------------------------------------------------------
# 2) Filter rows containing AGN/QSO/Quasar/Seyfert/Blazar words
# -----------------------------------------------------------
KEYWORDS = ["agn", "qso", "quasar", "seyfert", "blazar"]

def filter_agn_quasar(df):
    mask = df['Label'].str.lower().apply(lambda x: any(k in x for k in KEYWORDS))
    return df[mask].reset_index(drop=True)

# -----------------------------------------------------------
# 3) Summarize ALeRCE cone search and light curve info
# -----------------------------------------------------------
def summarize_matches(df, radius_arcsec=30):
    summary = []

    for i, row in df.iterrows():
        ra = row['RAdeg']
        dec = row['DEdeg']
        label = row['Label']

        # Perform cone search
        try:
            result = alerce.catshtm_conesearch(
                ra=ra, dec=dec, radius=radius_arcsec, format='pandas'
            )
        except:
            result = {}

        matches = []
        for cat, cat_df in result.items():
            if isinstance(cat_df, pd.DataFrame):
                matches.append(cat_df)

        if matches:
            matches_df = pd.concat(matches, ignore_index=True)
        else:
            matches_df = pd.DataFrame()

        # Pull object IDs
        oids = []
        for col in ['oid', 'objectId', 'object_id', 'objid', 'id']:
            if col in matches_df.columns:
                oids = matches_df[col].dropna().astype(str).unique().tolist()
                break

        # Count detections in each object's light curve
        lc_counts = []
        for oid in oids:
            try:
                lc = alerce.query_lightcurve(oid, format='pandas')
                lc_df = lc if isinstance(lc, pd.DataFrame) else pd.DataFrame(lc.get('detections', []))
                lc_counts.append(len(lc_df))
            except:
                lc_counts.append(0)

        summary.append({
            "Rank": row["Rank"],
            "Source ID": row["ID"],
            "Label": label,
            "RA (deg)": ra,
            "Dec (deg)": dec,
            "Num ZTF Matches": len(oids),
            "ZTF Object IDs": ", ".join(oids) if oids else "",
            "Light Curve # Points": ", ".join(map(str, lc_counts)) if lc_counts else ""
        })

    return pd.DataFrame(summary)

# -----------------------------------------------------------
# 4) Run the pipeline
# -----------------------------------------------------------
url = "https://content.cld.iop.org/journals/2041-8205/956/1/L6/revision1/apjlacfa03t1_mrt.txt"
df0 = load_table(url)
df_filt = filter_agn_quasar(df0)

print(f"Detected {len(df_filt)} AGN/QSO-like objects in the input table.")

summary_table = summarize_matches(df_filt)

# Show table in notebook/console
print(summary_table)

# Save to CSV
summary_table.to_csv("agn_ztf_summary.csv", index=False)
print("\nSaved summary to agn_ztf_summary.csv ✅")

Detected 49 AGN/QSO-like objects in the input table.
   Rank          Source ID                                   Label  \
0     2  39633127651410738             AGN, Velocity Structure, WR   
1     5  39633158374688229                                  Quasar   
2     6  39633149658924999             AGN, Velocity Structure, WR   
3     9  39627794417717430         AGN, Velocity Structure, NI5200   
4    13  39627806434395296             AGN, Velocity Structure, WR   
5    15  39627782380067362                                  Quasar   
6    22  39633278931569179                                  Quasar   
7    23  39632981354087949                                  Quasar   
8    29  39633290071641162                              BAL Quasar   
9    33  39632936152072650             AGN, Velocity Structure, WR   
10   41  39633140855081990             AGN, Velocity Structure, WR   
11   44  39633308396555667             AGN, Velocity Structure, WR   
12   51  39627782338122510     AGN, V

In [1]:
import warnings
try:
    from astropy.units import UnitsWarning
except ImportError:
    from astropy.utils.exceptions import UnitsWarning
warnings.filterwarnings("ignore", category=UnitsWarning)

import pandas as pd
import io
import requests
from alerce.core import Alerce
import matplotlib.pyplot as plt
import os

alerce = Alerce()

# Create output directory for light curves
output_dir = "lightcurves"
os.makedirs(output_dir, exist_ok=True)

# Load the table
def load_table(url):
    txt = requests.get(url).text
    colspecs = [(0,3), (4,21), (22,34), (35,46), (47,52), (53,91), (92,101)]
    names = ['Rank','ID','RAdeg','DEdeg','-logp','Label','Ref']
    df = pd.read_fwf(io.StringIO(txt), colspecs=colspecs, names=names, skiprows=3)
    df['RAdeg'] = pd.to_numeric(df['RAdeg'], errors='coerce')
    df['DEdeg'] = pd.to_numeric(df['DEdeg'], errors='coerce')
    df['Label'] = df['Label'].fillna('')
    return df

# Filter AGN/Quasar
def filter_agn_quasar(df):
    mask = df['Label'].str.contains(r'\bAGN\b|\bQuasar\b', case=False, na=False, regex=True)
    return df[mask].reset_index(drop=True)

# Fetch ALeRCE matches
def fetch_matches(ra_deg, dec_deg, radius_arcsec=1.0):
    result = alerce.catshtm_conesearch(ra=ra_deg, dec=dec_deg, radius=radius_arcsec, format='pandas')
    matches = []
    for cat, df in result.items():
        if isinstance(df, pd.DataFrame) and not df.empty:
            df2 = df.copy()
            df2['catalog'] = cat
            matches.append(df2)
    if matches:
        return pd.concat(matches, ignore_index=True)
    else:
        return pd.DataFrame()  # empty

# Fetch and plot light curve
def fetch_and_plot_lightcurve(oid, save=True):
    try:
        lc = alerce.query_lightcurve(oid, format='pandas')
        if lc.empty:
            return None
        fig, ax = plt.subplots(figsize=(8,4))
        ax.errorbar(lc['mjd'], lc['magpsf'], yerr=lc.get('sigmapsf'), fmt='o', alpha=0.7)
        ax.invert_yaxis()
        ax.set_xlabel('MJD')
        ax.set_ylabel('mag')
        ax.set_title(f"Light curve for {oid}")
        ax.grid(True)
        if save:
            filename = os.path.join(output_dir, f"{oid}_lightcurve.png")
            plt.savefig(filename, dpi=150)
            plt.close(fig)
        else:
            plt.show()
        return lc
    except Exception as e:
        print(f"Error fetching light curve for {oid}: {e}")
        return None

if __name__ == "__main__":
    url = "https://content.cld.iop.org/journals/2041-8205/956/1/L6/revision1/apjlacfa03t1_mrt.txt"
    df0 = load_table(url)
    df_filt = filter_agn_quasar(df0)
    print(f"Found {len(df_filt)} AGN/Quasar candidates.\n")

    # Summary table
    summary = []

for i, row in df_filt.iterrows():
    ra, dec = row['RAdeg'], row['DEdeg']
    matches = fetch_matches(ra, dec)
    if not matches.empty:
        print(f"Found {len(matches)} match(es) for RA={ra}, Dec={dec}")
        # show first few columns for debugging
        print(matches.head())

        for _, m in matches.iterrows():
            # Try to get the object ID safely
            oid = None
            for col in ['oid', 'object_id', 'objid', 'id']:
                if col in m and pd.notna(m[col]):
                    oid = m[col]
                    break
            if oid is None:
                print("  -> No valid object ID found; skipping this match")
                continue

            print(f"Fetching light curve for {oid} ...")
            fetch_and_plot_lightcurve(oid, save=True)
            summary.append({
                'Rank': row['Rank'],
                'ID': row['ID'],
                'RAdeg': ra,
                'DEdeg': dec,
                'Label': row['Label'],
                'ALeRCE_ID': oid,
                'Catalog': m['catalog']
            })
    else:
        print(f"No matches found for RA={ra}, Dec={dec}")
        summary.append({
            'Rank': row['Rank'],
            'ID': row['ID'],
            'RAdeg': ra,
            'DEdeg': dec,
            'Label': row['Label'],
            'ALeRCE_ID': None,
            'Catalog': None
        })

    summary_df = pd.DataFrame(summary)
    print("\n=== Summary Table ===")
    print(summary_df)

    # Optional: save the summary table to CSV
    summary_df.to_csv("AGN_Quasar_ALeRCE_summary.csv", index=False)

Found 49 AGN/Quasar candidates.

No matches found for RA=241.19113199, Dec=42.84971658

=== Summary Table ===
  Rank                 ID       RAdeg      DEdeg                        Label  \
0    2  39633127651410738  241.191132  42.849717  AGN, Velocity Structure, WR   

  ALeRCE_ID Catalog  
0      None    None  
No matches found for RA=245.13714079, Dec=44.38041175

=== Summary Table ===
  Rank                 ID       RAdeg      DEdeg                        Label  \
0    2  39633127651410738  241.191132  42.849717  AGN, Velocity Structure, WR   
1    5  39633158374688229  245.137141  44.380412                       Quasar   

  ALeRCE_ID Catalog  
0      None    None  
1      None    None  
Found 3 match(es) for RA=241.46246538, Dec=44.09456197
   Detector  ExposureTime   ImageID  KronRadius   MagAper2    MagAuto  \
0       3.0         480.0  373077.0       0.455  19.617090  19.292490   
1       3.0         900.0  373077.0       0.455  21.135191  20.812691   
2       3.0         40